In [1]:
import os
from gc import callbacks

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from dotenv import load_dotenv, find_dotenv
from sqlalchemy import create_engine
from sqlalchemy.engine import URL
import phik
from sklearn.base import clone
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, TargetEncoder, OrdinalEncoder, OneHotEncoder
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import (
    roc_auc_score, average_precision_score, log_loss, brier_score_loss,
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, ConfusionMatrixDisplay,)
from sklearn.calibration import calibration_curve
import lightgbm as lgb
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
import optuna
from optuna.trial import FixedTrial
import math
import category_encoders as ce


load_dotenv(find_dotenv(usecwd=True))
url = URL.create("postgresql+psycopg2",
    username=os.getenv("DB_USER"), password=os.getenv("DB_PASSWORD"),
    host="localhost", port=5433, database="hotel")
engine = create_engine(url)

df = pd.read_sql("SELECT * FROM hotels", engine)

print(df.shape)
df = df.drop_duplicates().reset_index(drop=True)
print(df.shape)
print(df['is_canceled'].mean())

/home/artem/hotel_book/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(119390, 32)
(87396, 32)
0.27489816467572886


In [2]:
df["total_night"] = df["stays_in_weekend_nights"] + df["stays_in_week_nights"]
df = df.drop(columns=["company","assigned_room_type","distribution_channel", "reservation_status", "reservation_status_date", "stays_in_weekend_nights", "stays_in_week_nights", "arrival_date_year", "arrival_date_month"])
df['country'] = df['country'].fillna("PRT")
df['agent'] = df['agent'].fillna(0)
df = df[df['lead_time'] < 600]
df = df[(df['adr'] < 5000) & (df['adr'] >= 0)]
df["adr_zero"] = (df["adr"] == 0).astype(int)
df = df[df['total_night'] < 31]
df["children"] = pd.to_numeric(df["children"], errors="coerce")
df = df.dropna(subset=["children"]).reset_index(drop=True)
df['children'] = df['children'].astype(int)


In [3]:
x = df.drop(columns='is_canceled')
y = df['is_canceled']

x_tmp, x_test, y_tmp, y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)
print(x_tmp.shape, y_tmp.shape)
x_tr, x_val, y_tr, y_val = train_test_split(x_tmp, y_tmp, test_size=0.25, random_state=42, stratify=y_tmp)


(69882, 24) (69882,)


60% train, 20% val, 20% test

In [4]:
order = ["No Deposit", "Non Refund", "Refundable"]
prep = ColumnTransformer(
    [
        ("hotel", OneHotEncoder(drop="if_binary", sparse_output=False), ["hotel"]),
        ("cust", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["customer_type"]),
        ("deposit", OrdinalEncoder(categories=[order], handle_unknown="use_encoded_value",
                                   unknown_value=-1), ["deposit_type"]),
        ("meal", ce.CountEncoder(normalize=True), ["meal"]),
        ("country", ce.BinaryEncoder(), ["country"]),
        ("market_segment", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["market_segment"]),
        ("reserved_room_type", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["reserved_room_type"]),

    ],
    remainder="passthrough",
)

In [5]:
prep.set_params(verbose_feature_names_out=False)
prep.set_output(transform="pandas")

x_tr_t = prep.fit_transform(x_tr)
x_val_t = prep.transform(x_val)
x_test_t = prep.transform(x_test)
print(x_tr_t.head(5).T)
print(x_val_t.shape)

                                    59012     26438       30694      31027  \
hotel_Resort Hotel               0.000000   1.00000    1.000000    1.00000   
customer_type_Contract           0.000000   0.00000    0.000000    0.00000   
customer_type_Group              0.000000   0.00000    0.000000    0.00000   
customer_type_Transient          1.000000   1.00000    1.000000    1.00000   
customer_type_Transient-Party    0.000000   0.00000    0.000000    0.00000   
deposit_type                     0.000000   0.00000    0.000000    0.00000   
meal                             0.107535   0.77644    0.106428    0.77644   
country_0                        0.000000   0.00000    0.000000    0.00000   
country_1                        0.000000   0.00000    0.000000    0.00000   
country_2                        0.000000   0.00000    0.000000    0.00000   
country_3                        0.000000   0.00000    0.000000    0.00000   
country_4                        0.000000   0.00000    0.000000 

In [6]:
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(x_tr, y_tr)
print(roc_auc_score(y_test, dummy.predict_proba(x_test)[:, 1]))
print(accuracy_score(y_test, dummy.predict(x_test)))

0.5
0.7252017629214126


baseline - most_frequent

In [7]:
print(df.groupby("deposit_type")["is_canceled"].agg(n="count", cancel_share="mean").round(3))

                  n  cancel_share
deposit_type                     
No Deposit    86221         0.267
Non Refund     1025         0.946
Refundable      107         0.243


очень старнная аномалия, в 95% людей с невозратным дипозитом отменяют бронь, пока я оставлю этот бред, потом удалю при первой возможности

In [ ]:
def unic(model_name, trial):
    if model_name == 'lr':
        params = {"C": trial.suggest_float("lr_C", 1e-4, 1e2, log=True),
                  "penalty": trial.suggest_categorical("lr_penalty", ["l1", "l2"]),
                  "solver": "liblinear",
                  "class_weight": trial.suggest_categorical("lr_class_weight", [None, "balanced"]),
                  "max_iter": 3000,
                  "random_state": 42}
        model = Pipeline([("scaler", StandardScaler()), ("clf", LogisticRegression(**params))])
    elif model_name == 'rf':
        params = {"n_estimators": trial.suggest_int("rf_n_estimators", 100, 2000),
                  "max_depth": trial.suggest_int("rf_max_depth", 2, 11),
                  "random_state": 42,
                  "min_samples_split": trial.suggest_int("rf_min_samples_split", 2, 60),
                  "min_samples_leaf": trial.suggest_int("rf_min_samples_leaf", 1, 60),
                  "max_features": trial.suggest_float("rf_max_features", 0.2, 1.0),
                  "max_samples": trial.suggest_float("rf_max_samples", 0.5, 1.0),
                  "class_weight": trial.suggest_categorical("rf_class_weight", [None, "balanced", "balanced_subsample"]),
                  "bootstrap": True,
                  "n_jobs": -1}
        model = RandomForestClassifier(**params)
    elif model_name == 'xgb':
        params = {"n_estimators": 3000,
                  "max_depth": trial.suggest_int("xgb_max_depth", 3, 11),
                  "random_state": 42,
                  "learning_rate": trial.suggest_float("xgb_learning_rate", 0.0001, 1.0, log=True),
                  "subsample": trial.suggest_float("xgb_subsample", 0.6, 1.0),
                  "colsample_bytree": trial.suggest_float("xgb_colsample_bytree", 0.3, 1.0),
                  "min_child_weight": trial.suggest_int("xgb_min_child_weight",1, 30, log=True),
                  "reg_alpha": trial.suggest_float("xgb_reg_alpha", 1e-3, 10.0, log=True),
                  "reg_lambda": trial.suggest_float("xgb_reg_lambda", 1e-3, 10.0, log=True),
                  "eval_metric": ["auc", "error"],
                  "early_stopping_rounds": 100,
                  "tree_method": "hist",
                  "n_jobs": -1,
                  "objective": "binary:logistic",
                  "max_bin": trial.suggest_int("xgb_max_bin", 64, 512),}
        model = XGBClassifier(**params)
    elif model_name == 'lgb':
        params = {"n_estimators": 3000,
                  "learning_rate": trial.suggest_float("lgb_learning_rate", 0.0001, 1.0, log=True),
                  "num_leaves": trial.suggest_int("lgb_num_leaves", 8, 256, log=True),
                  "min_child_samples": trial.suggest_int("lgb_min_child_samples", 5, 120),
                  "subsample": trial.suggest_float("lgb_subsample", 0.6, 1.0),
                  "subsample_freq": 1,
                  "colsample_bytree": trial.suggest_float("lgb_colsample_bytree", 0.3, 1.0),
                  "reg_alpha": trial.suggest_float("lgb_reg_alpha", 1e-3, 10.0, log=True),
                  "reg_lambda": trial.suggest_float("lgb_reg_lambda", 1e-3, 10.0, log=True),
                  "metric": ['auc', 'error'], # доработка, возможен подбор и для log loss
                  "random_state": 42,
                  "n_jobs": -1,
                  "verbosity": -1,
                  "early_stopping_rounds": 100,}
        model = LGBMClassifier(**params)
    elif model_name == 'cat':
        params = {"n_estimators": 3000,
                  "learning_rate": trial.suggest_float("cat_learning_rate", 0.0001, 1.0, log=True),
                  "depth": trial.suggest_int("cat_depth", 2, 11),
                  "l2_leaf_reg": trial.suggest_float("cat_l2_leaf_reg", 1e-2, 10.0, log=True),
                  "random_strength": trial.suggest_float("cat_random_strength", 1e-3, 7.0, log=True),
                  "border_count": trial.suggest_int("cat_border_count", 32, 200),
                  "eval_metric": "AUC",
                  "early_stopping_rounds": 100,
                  "random_seed": 42,
                  "thread_count": -1,
                  "verbose": False,
                  "custom_metric": ["Accuracy", "F1"],
                  "loss_function": "Logloss",}
        bootstrap = trial.suggest_categorical("cat_bootstrap", ["Bayesian", "Bernoulli", "MVS"])
        params["bootstrap_type"] = bootstrap
        if bootstrap == "Bayesian":
            params["bagging_temperature"] = trial.suggest_float("cat_bagging_temp", 0.0, 1.0)
        else:
             params["subsample"] = trial.suggest_float("cat_subsample", 0.5, 1.0)
        params["rsm"] = trial.suggest_float("cat_rsm", 0.0, 1.0)
        model = CatBoostClassifier(**params)



In [ ]:
def fit_model(model, name, x_tr, y_tr, x_val, y_val):
    if name == 'xgb':
        model.fit(x_tr, y_tr, eval_set=[(x_val, y_val)], verbose=False)
        return model.best_iteration + 1
    if name == 'lgb':
        model.fit(x_tr, y_tr, eval_set=[(x_val, y_val)], callbacks=[lgb.early_stopping(100, first_metric_only=True, verbose=False), lgb.log_evaluation(0)])
        return model.best_iteration_
    if name == 'cat':
        model.fit(x_tr, y_tr, eval_set=[(x_val, y_val)], verbose=False)
        return model.best_iteration() + 1
    model.fit(x_tr, y_tr)
    return None

In [ ]:
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
def make_cv(name):
    def objective(trial):
        scores, iters = [], []
        for tr_idx, val_idx in skf.split(x_tmp, y_tmp):
            xa, xb, = x_tmp.iloc[tr_idx], x_tmp.iloc[val_idx]
            ya, yb = y_tmp.iloc[tr_idx], y_tmp.iloc[val_idx]
            model = unic(trial, name)
            n_trees = fit_model(model, name, xa, ya, xb, yb)
            proba = model.predict_proba(xb)[:, 1]
            scores.append(roc_auc_score(yb, proba))
            if n_trees is not None:
                iters.append(n_trees)
        if iters:
            trial.set_user_attr("best_iter", int(np.mean(iters)))
        trial.set_user_attr("auc_std", float(np.std(scores)))
    return objective